# SIGNOVA — KArSL 502-class GRU baseline

Run this notebook in **Google Colab with a GPU runtime** after the feature preparation has completed. It reads the frozen feature shards from your existing private Drive folder. It trains on `PASS` samples in `train`, selects a checkpoint using `PASS` samples in `validation`, and never loads the `test` shards. `REVIEW` and `FAILED` samples remain recorded in the source audit and are excluded from this baseline.

This is a new 502-class experiment; the 30-class checkpoint and its accuracy do not transfer. The result is a research baseline, not a verified sign-language translator. Label meanings and the conflicting 27/28 test labels still need manual review before final test evaluation.


## 1. Mount Drive and verify the frozen source

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import os
import random
import hashlib
import shutil
import numpy as np
import torch

SOURCE = Path('/content/drive/MyDrive/SIGNOVA/KArSL/ml_expansion/artifacts/private/features_502_v1')
OUTPUT = Path('/content/drive/MyDrive/SIGNOVA/KArSL/ml_expansion/experiments/gru_502_pass_v1')
CACHE = Path('/content/signova_502_train_cache')
EXPECTED_FINGERPRINT = '5d0d43950dec9c69bef2766e48550463a027b49a44748631b976f211c77e7c4a'
PREPROCESSING_VERSION = 'pose_v1_32x50_xy_visibility_shoulder_conf05'
SEED = 42

assert SOURCE.is_dir(), f'Feature directory not found: {SOURCE}'
report = json.loads((SOURCE / 'preprocessing_report.json').read_text())
manifest = json.loads((SOURCE / 'shards.json').read_text())
assert report['fingerprint'] == manifest['fingerprint'] == EXPECTED_FINGERPRINT
assert report['expected_samples'] == 75515
assert report['source_classes'] == report['prepared_classes'] == 502
assert report['feature_shape'] == [32, 150]
assert (report['technical_pass_samples'], report['review_samples'], report['failed_samples']) == (75392, 119, 4)
assert len(manifest['shards']) == 149
assert all(x['split'] in {'train', 'validation', 'test'} for x in manifest['shards'])

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
print('Source:', SOURCE)
print('Frozen fingerprint:', EXPECTED_FINGERPRINT)
print('Training and validation shards:', sum(x['split'] != 'test' for x in manifest['shards']))
print('Test shards will not be loaded.')


Mounted at /content/drive
GPU: Tesla T4
Source: /content/drive/MyDrive/SIGNOVA/KArSL/ml_expansion/artifacts/private/features_502_v1
Frozen fingerprint: 5d0d43950dec9c69bef2766e48550463a027b49a44748631b976f211c77e7c4a
Training and validation shards: 99
Test shards will not be loaded.


## 2. Copy and verify train/validation shards, then filter by the embedded QC audit

The local `/content` cache speeds up reads. It can be recreated after a Colab restart. The original shards on Drive are read only. SHA-256 and sample/label/path checks stop training if the source has changed.

In [ ]:
from collections import Counter

def file_sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def load_pass_split(split):
    assert split in ('train', 'validation'), 'The test split is deliberately unavailable here.'
    records = [r for r in manifest['shards'] if r['split'] == split]
    assert records
    xs, ys, paths, signer_ids = [], [], [], []
    qc = Counter()

    for item in records:
        relative = Path(item['path'])
        assert relative.parts[0] == split and relative.suffix == '.npz'
        original = SOURCE / relative
        local = CACHE / relative
        local.parent.mkdir(parents=True, exist_ok=True)
        if not local.is_file() or file_sha256(local) != item['sha256']:
            shutil.copy2(original, local)
        assert file_sha256(local) == item['sha256'], f'Checksum mismatch: {relative}'

        with np.load(local, allow_pickle=False) as part:
            assert str(part['fingerprint'].item()) == EXPECTED_FINGERPRINT
            assert str(part['preprocessing_version'].item()) == PREPROCESSING_VERSION
            assert str(part['split'].item()) == split
            audit = json.loads(str(part['audit_json'].item()))
            assert len(audit) == item['expected_samples']
            qc.update(row['status'] for row in audit)
            prepared = [row for row in audit if row['status'] != 'FAILED']
            x = part['X']
            labels = part['labels']
            sample_paths = part['paths'].astype(str)
            assert x.shape == (item['prepared_samples'], 32, 150)
            assert labels.shape == sample_paths.shape == (len(prepared),)
            assert x.dtype == np.float32 and np.isfinite(x).all()
            assert sample_paths.tolist() == [row['pose_file'] for row in prepared]
            assert labels.tolist() == [row['sign_id'] for row in prepared]
            keep = np.array([row['status'] == 'PASS' for row in prepared])
            xs.append(x[keep])
            ys.append(labels[keep].astype(np.int64))
            paths.extend(sample_paths[keep].tolist())
            signer_ids.extend(row['signer_id'] for row in prepared if row['status'] == 'PASS')

    X = np.concatenate(xs)
    y = np.concatenate(ys)
    assert len(paths) == len(set(paths)) == len(X) == len(y)
    assert set(y) == set(range(1, 503)), f'{split}: missing classes'
    assert qc['PASS'] == len(y)
    return X, y, paths, set(signer_ids), qc


X_train, y_train, train_paths, train_signers, train_qc = load_pass_split('train')
X_val, y_val, val_paths, val_signers, val_qc = load_pass_split('validation')
assert train_signers.isdisjoint(val_signers), 'Signer leakage between train and validation'
assert set(train_paths).isdisjoint(val_paths), 'Sample leakage between train and validation'
assert (len(X_train), train_qc['REVIEW'], train_qc['FAILED']) == (25032, 55, 0)
assert (len(X_val), val_qc['REVIEW'], val_qc['FAILED']) == (25254, 58, 4)

print('Train PASS:', len(X_train), '| REVIEW excluded:', train_qc['REVIEW'])
print('Validation PASS:', len(X_val), '| REVIEW excluded:', val_qc['REVIEW'], '| FAILED:', val_qc['FAILED'])
print('Classes in each split:', len(set(y_train)), len(set(y_val)))
print('Signer groups:', sorted(train_signers), sorted(val_signers))
print('Test data accessed: NO')


Train PASS: 25032 | REVIEW excluded: 55
Validation PASS: 25254 | REVIEW excluded: 58 | FAILED: 4
Classes in each split: 502 502
Signer groups: [3] [2]
Test data accessed: NO


## 3. Model and evaluation setup

This bidirectional GRU baseline uses the frozen 32 × 150 pose features. Labels 1–502 are mapped deterministically to output indices 0–501. Macro F1 on validation selects the best checkpoint; test is reserved for a later audited evaluation.

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score

class SignGRU502(nn.Module):
    def __init__(self, hidden_size=128, num_classes=502):
        super().__init__()
        self.gru = nn.GRU(150, hidden_size, num_layers=2, batch_first=True,
                          bidirectional=True, dropout=0.25)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size * 4),
            nn.Dropout(0.30),
            nn.Linear(hidden_size * 4, num_classes),
        )

    def forward(self, x):
        sequence, _ = self.gru(x)
        pooled = torch.cat((sequence.mean(dim=1), sequence.amax(dim=1)), dim=1)
        return self.head(pooled)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SignGRU502().to(device)
batch_size = 128
train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train),
                                       torch.from_numpy(y_train - 1)),
                          batch_size=batch_size, shuffle=True,
                          num_workers=0, pin_memory=(device.type == 'cuda'))
val_loader = DataLoader(TensorDataset(torch.from_numpy(X_val),
                                     torch.from_numpy(y_val - 1)),
                        batch_size=256, shuffle=False, num_workers=0,
                        pin_memory=(device.type == 'cuda'))
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.001)
loss_fn = nn.CrossEntropyLoss()

def validate():
    model.eval()
    truth, predictions, probabilities = [], [], []
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            logits = model(batch_x.to(device, non_blocking=True))
            probs = logits.softmax(dim=1).cpu().numpy()
            probabilities.append(probs)
            truth.append(batch_y.numpy())
            predictions.append(probs.argmax(axis=1))
    y = np.concatenate(truth)
    pred = np.concatenate(predictions)
    prob = np.concatenate(probabilities)
    return {
        'accuracy': float(accuracy_score(y, pred)),
        'macro_f1': float(f1_score(y, pred, average='macro', labels=np.arange(502), zero_division=0)),
        'top5_accuracy': float(top_k_accuracy_score(y, prob, k=5, labels=np.arange(502))),
    }

print('Model parameters:', sum(p.numel() for p in model.parameters()))
print('Device:', device)


Model parameters: 770038
Device: cuda


## 4. Train and save the best validation checkpoint

Run with a GPU runtime. The checkpoint and history are saved in a new experiment folder on Drive; the older 30-class model stays untouched. Re-running this cell starts a fresh experiment and overwrites only this experiment's outputs.

In [ ]:
import pandas as pd
from datetime import datetime, timezone

assert device.type == 'cuda', 'Switch Colab runtime to GPU, then rerun cells 1–4.'
OUTPUT.mkdir(parents=True, exist_ok=True)
best_path = OUTPUT / 'best_gru_502_pass_v1.pt'
history_path = OUTPUT / 'history.csv'
metadata_path = OUTPUT / 'experiment.json'

metadata = {
    'experiment': 'gru_502_pass_v1',
    'started_utc': datetime.now(timezone.utc).isoformat(),
    'source_fingerprint': EXPECTED_FINGERPRINT,
    'preprocessing_version': PREPROCESSING_VERSION,
    'classes': list(range(1, 503)),
    'train_pass': len(X_train), 'validation_pass': len(X_val),
    'train_review_excluded': train_qc['REVIEW'],
    'validation_review_excluded': val_qc['REVIEW'],
    'validation_failed': val_qc['FAILED'],
    'test_accessed': False,
    'label_semantics_verified': False,
    'known_test_issue': '50 identical pairs labeled 27 and 28; historical test use recorded',
    'seed': SEED, 'batch_size': batch_size, 'epochs_max': 25,
    'selection_metric': 'validation_macro_f1',
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

best_f1 = -1.0
best_epoch = 0
stale_epochs = 0
history = []

for epoch in range(1, 26):
    model.train()
    total_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_x)
        loss = loss_fn(logits, batch_y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(batch_y)

    metrics = validate()
    row = {'epoch': epoch, 'train_loss': total_loss / len(X_train), **metrics}
    history.append(row)
    pd.DataFrame(history).to_csv(history_path.with_suffix('.tmp'), index=False)
    os.replace(history_path.with_suffix('.tmp'), history_path)
    print(f"Epoch {epoch:02d}: loss={row['train_loss']:.4f}, "
          f"val_acc={row['accuracy']:.4f}, val_macro_f1={row['macro_f1']:.4f}, "
          f"val_top5={row['top5_accuracy']:.4f}", flush=True)

    if metrics['macro_f1'] > best_f1 + 1e-6:
        best_f1, best_epoch, stale_epochs = metrics['macro_f1'], epoch, 0
        checkpoint = {
            'model_state_dict': model.state_dict(),
            'model_name': 'SignGRU502',
            'model_config': {'hidden_size': 128, 'num_classes': 502},
            'input_shape': [32, 150],
            'classes': list(range(1, 503)),
            'source_fingerprint': EXPECTED_FINGERPRINT,
            'preprocessing_version': PREPROCESSING_VERSION,
            'selection_metric': 'validation_macro_f1',
            'best_epoch': best_epoch,
            'validation_metrics': metrics,
            'test_evaluated': False,
        }
        temporary = best_path.with_suffix('.tmp')
        torch.save(checkpoint, temporary)
        os.replace(temporary, best_path)
    else:
        stale_epochs += 1
        if stale_epochs >= 6:
            print('Early stopping after six epochs without macro F1 improvement.')
            break

assert best_path.is_file()
saved = torch.load(best_path, map_location='cpu', weights_only=True)
assert saved['source_fingerprint'] == EXPECTED_FINGERPRINT
assert saved['classes'] == list(range(1, 503))
print('Best epoch:', saved['best_epoch'])
print('Best validation metrics:', saved['validation_metrics'])
print('Saved checkpoint:', best_path)
print('Saved history:', history_path)
print('Test evaluated: NO')


Epoch 01: loss=3.1708, val_acc=0.2170, val_macro_f1=0.1853, val_top5=0.4436
Epoch 02: loss=0.5907, val_acc=0.2921, val_macro_f1=0.2481, val_top5=0.5193
Epoch 03: loss=0.2268, val_acc=0.3104, val_macro_f1=0.2741, val_top5=0.5492
Epoch 04: loss=0.1370, val_acc=0.3133, val_macro_f1=0.2750, val_top5=0.5446
Epoch 05: loss=0.0988, val_acc=0.3293, val_macro_f1=0.2896, val_top5=0.5686
Epoch 06: loss=0.0797, val_acc=0.3270, val_macro_f1=0.2856, val_top5=0.5590
Epoch 07: loss=0.0669, val_acc=0.3368, val_macro_f1=0.2989, val_top5=0.5677
Epoch 08: loss=0.0583, val_acc=0.3503, val_macro_f1=0.3119, val_top5=0.5711
Epoch 09: loss=0.0413, val_acc=0.3277, val_macro_f1=0.2914, val_top5=0.5595
Epoch 10: loss=0.0485, val_acc=0.3442, val_macro_f1=0.3070, val_top5=0.5860
Epoch 11: loss=0.0443, val_acc=0.3481, val_macro_f1=0.3074, val_top5=0.5822
Epoch 12: loss=0.0429, val_acc=0.3618, val_macro_f1=0.3247, val_top5=0.5893
Epoch 13: loss=0.0374, val_acc=0.3450, val_macro_f1=0.3152, val_top5=0.5851
Epoch 14: lo

## What the result means

The printed scores are **validation** results. Do not call them test accuracy. The 50 exact duplicate pairs with conflicting labels 27/28 occur in the test split, and 450 test samples were used in older experiments. Review the label glossary/source for classes 27 and 28 and define a separate evaluation policy before evaluating this checkpoint on test. The `REVIEW` items also need manual inspection before deciding whether to include them in a later training version. Do not automatically relabel or delete source files.


In [ ]:
from pathlib import Path
import hashlib
import subprocess
import sys

script = Path(
    "/content/drive/MyDrive/SIGNOVA/KArSL/ml_expansion/scripts/"
    "signova_compare_cnn_502.py"
)

assert script.is_file(), f"Script not found: {script}"
assert hashlib.sha256(script.read_bytes()).hexdigest() == (
    "f6e0f60a817de984fc356b7bbd7edcaff72ffbc63e046fe09175564cc6633b5e"
), "Script checksum mismatch."

process = subprocess.Popen(
    [sys.executable, "-u", str(script)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
except KeyboardInterrupt:
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    raise

if return_code != 0:
    raise RuntimeError("Comparison stopped. Review the output above.")

Loading the frozen train/validation PASS samples.
train: verified 10/49 shards
train: verified 20/49 shards
train: verified 30/49 shards
train: verified 40/49 shards
train: verified 49/49 shards
validation: verified 10/50 shards
validation: verified 20/50 shards
validation: verified 30/50 shards
validation: verified 40/50 shards
validation: verified 50/50 shards
GRU checkpoint reload and full validation: PASS
{
  "accuracy": 0.3775639502653045,
  "macro_f1": 0.342525449074227,
  "top5_accuracy": 0.599192207175101
}
Train PASS: 25032 | Validation PASS: 25254 | Classes: 502
Model: SignCNN502 | Parameters: 576886
Device: cuda | Test features loaded: NO
Epoch 01: loss=2.8457, val_acc=0.2093, val_macro_f1=0.1750, val_top5=0.4343
Epoch 02: loss=0.6747, val_acc=0.2614, val_macro_f1=0.2229, val_top5=0.4899
Epoch 03: loss=0.3206, val_acc=0.2903, val_macro_f1=0.2581, val_top5=0.5342
Epoch 04: loss=0.2175, val_acc=0.3093, val_macro_f1=0.2762, val_top5=0.5565
Epoch 05: loss=0.1575, val_acc=0.3268,

In [ ]:
from pathlib import Path
import hashlib
import subprocess
import sys

script = Path(
    "/content/drive/MyDrive/SIGNOVA/KArSL/ml_expansion/scripts/"
    "signova_calibrate_cnn_502.py"
)

assert script.is_file(), f"Script not found: {script}"
assert hashlib.sha256(script.read_bytes()).hexdigest() == (
    "a83f47c0de8d3d0192cbc18dacbfc7f5cee5fc95bbe3b0ae26e1b42d0a5fbbad"
), "Script checksum mismatch."

process = subprocess.Popen(
    [sys.executable, "-u", str(script)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
except KeyboardInterrupt:
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    raise

if return_code != 0:
    raise RuntimeError("Calibration stopped. Review the output above.")

Selected model: SignCNN502 | Epoch: 12 | Device: cuda
Protocol frozen: validation only; no test feature loading.
validation: verified 10/50 shards
validation: verified 20/50 shards
validation: verified 30/50 shards
validation: verified 40/50 shards
validation: verified 50/50 shards
temperature_fit: 12557 samples
threshold_select: 6053 samples
policy_audit: 6644 samples
CNN checkpoint reload and validation metrics: PASS

=== CALIBRATION RESULT ===
Temperature: 2.414195
Audit ECE before: 0.347401
Audit ECE after: 0.046079
Candidate threshold: None
Acceptance enabled: False | Policy status: NO_THRESHOLD_MET_SELECTION_TARGET
Candidate policy audit: {"samples": 6644, "threshold": null, "accepted": 0, "coverage": 0.0, "accepted_correct": 0, "accepted_accuracy": null, "accepted_error_rate": null, "wilson_95_lower": null}
Class reports and calibration files: /content/drive/MyDrive/SIGNOVA/KArSL/ml_expansion/experiments/cnn_502_pass_v1/calibration_v1
Checkpoint: UNCHANGED
Test evaluated: NO
OOV

In [2]:
from pathlib import Path
import hashlib
import subprocess
import sys

script = Path(
    "/content/drive/MyDrive/SIGNOVA/KArSL/ml_expansion/scripts/"
    "signova_evaluate_test_502.py"
)

assert script.is_file(), f"Script not found: {script}"
assert hashlib.sha256(script.read_bytes()).hexdigest() == (
    "cf5b1010f217735bb7451be8161fc41db4b7cbfa3607e03a0bed23bf42b059e4"
), "Script checksum mismatch."

process = subprocess.Popen(
    [sys.executable, "-u", str(script)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
except KeyboardInterrupt:
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    raise

if return_code != 0:
    raise RuntimeError("Test evaluation stopped. Review the output above.")

Test protocol: FROZEN before predictions
Primary: 24536 samples | True classes: 500 | Model outputs: 502
Secondary: 24956 samples | Contains historical test exposure
Acceptance: DISABLED | Calibration and model: FROZEN
test: verified 10/50 shards
test: verified 20/50 shards
test: verified 30/50 shards
test: verified 40/50 shards
test: verified 50/50 shards
Test predictions complete. Running fixed inference benchmarks.

=== TEST EVALUATION RESULT ===
Model: SignCNN502 | Output classes: 502 | Evaluated true classes: 500
Primary samples: 24536
Accuracy: 52.5187%
Macro F1: 0.485208
Top-5 Accuracy: 74.2908%
ECE before: 0.265556
ECE after: 0.111822
Active-policy coverage: 0.00% | Accepted accuracy: N/A
Abstention policy: UNCHANGED; all predictions remain uncertain.
Checkpoint size: 2.215 MiB
Benchmark cuda batch=1: mean=1.025 ms, p95=1.108 ms
Benchmark cuda batch=32: mean=1.261 ms, p95=1.353 ms
Benchmark cpu batch=1: mean=0.814 ms, p95=0.954 ms
Benchmark cpu batch=32: mean=9.266 ms, p95=14.9

In [3]:
from pathlib import Path
import hashlib
import subprocess
import sys

script = Path(
    "/content/drive/MyDrive/SIGNOVA/KArSL/ml_expansion/scripts/"
    "signova_predict_502.py"
)

assert script.is_file(), f"Script not found: {script}"
assert hashlib.sha256(script.read_bytes()).hexdigest() == (
    "06e4f036c547760f67744ee5d64f5b4fc023543cac8a910edb8ae9ff17fd14bb"
), "Script checksum mismatch."

candidates = [
    Path("/kaggle/input/kars502-pose-75/content/kars502-pose-75k/pose_dir/sign_0001_s2_sess2_014.pose"),
    Path("/root/.cache/kagglehub/datasets/tfmohamedyahia/kars502-pose-75/versions/1/content/kars502-pose-75k/pose_dir/sign_0001_s2_sess2_014.pose"),
]
real_pose = next((path for path in candidates if path.is_file()), None)

command = [sys.executable, "-u", str(script)]
if real_pose is not None:
    command.extend(["--pose-file", str(real_pose)])

result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Inference verification failed.")

if real_pose is None:
    print("Real Pose check: NOT RUN; source file is not present in this runtime.")

Model loading: PASS
Frozen preprocessing fingerprint: PASS
Synthetic Pose-to-features-to-prediction parity: PASS
Contract: PASS
{
  "status": "uncertain",
  "uncertain": true,
  "class_id": null,
  "label": null,
  "label_arabic": null,
  "confidence": 0.257777,
  "top_k": [
    {
      "class_id": 131,
      "label": "itch",
      "label_arabic": "حكة / هرش",
      "score": 0.257777,
      "label_review_status": "unverified",
      "label_conflict": false
    },
    {
      "class_id": 87,
      "label": "nervous system",
      "label_arabic": "جهاز عصبي",
      "score": 0.114222,
      "label_review_status": "unverified",
      "label_conflict": false
    },
    {
      "class_id": 268,
      "label": "treason",
      "label_arabic": "خيانة",
      "score": 0.064608,
      "label_review_status": "unverified",
      "label_conflict": false
    }
  ],
  "model_version": "karsl_cnn_502_pass_v1",
  "calibration_version": "cnn502_validation_temperature_v1"
}
Status: INFERENCE VERIFICATION

In [6]:
from pathlib import Path
import hashlib
import kagglehub
import subprocess
import sys

script = Path(
    "/content/drive/MyDrive/SIGNOVA/KArSL/ml_expansion/scripts/"
    "signova_predict_502.py"
)
assert hashlib.sha256(script.read_bytes()).hexdigest() == (
    "06e4f036c547760f67744ee5d64f5b4fc023543cac8a910edb8ae9ff17fd14bb"
)

pose_path = Path(
    kagglehub.dataset_download(
        "tfmohamedyahia/kars502-pose-75/versions/1",
        path="content/kars502-pose-75k/pose_dir/sign_0001_s2_sess2_014.pose",
    )
)
assert pose_path.is_file() and 0 < pose_path.stat().st_size < 10_000_000
print("Real Pose file:", pose_path.name)

result = subprocess.run(
    [sys.executable, "-u", str(script), "--pose-file", str(pose_path)],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("Real Pose inference check failed.")

Real Pose file: sign_0001_s2_sess2_014.pose
Model loading: PASS
Frozen preprocessing fingerprint: PASS
Synthetic Pose-to-features-to-prediction parity: PASS
Contract: PASS
{
  "status": "uncertain",
  "uncertain": true,
  "class_id": null,
  "label": null,
  "label_arabic": null,
  "confidence": 0.257777,
  "top_k": [
    {
      "class_id": 131,
      "label": "itch",
      "label_arabic": "حكة / هرش",
      "score": 0.257777,
      "label_review_status": "unverified",
      "label_conflict": false
    },
    {
      "class_id": 87,
      "label": "nervous system",
      "label_arabic": "جهاز عصبي",
      "score": 0.114222,
      "label_review_status": "unverified",
      "label_conflict": false
    },
    {
      "class_id": 268,
      "label": "treason",
      "label_arabic": "خيانة",
      "score": 0.064608,
      "label_review_status": "unverified",
      "label_conflict": false
    }
  ],
  "model_version": "karsl_cnn_502_pass_v1",
  "calibration_version": "cnn502_validation_temp

In [5]:
import subprocess
import sys
from pathlib import Path

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pose-format==0.15.0"],
    check=True,
)
subprocess.run(
    [sys.executable, "-c", "from pose_format import Pose; print('Pose decoder: PASS')"],
    check=True,
)

pose_path = Path(
    "/root/.cache/kagglehub/datasets/tfmohamedyahia/"
    "kars502-pose-75/versions/1/content/kars502-pose-75k/"
    "pose_dir/sign_0001_s2_sess2_014.pose"
)
assert pose_path.is_file(), "The downloaded Pose file is missing from this runtime."

result = subprocess.run(
    [
        sys.executable, "-u",
        "/content/drive/MyDrive/SIGNOVA/KArSL/ml_expansion/scripts/signova_predict_502.py",
        "--pose-file", str(pose_path),
    ],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("Real Pose inference check failed.")

Model loading: PASS
Frozen preprocessing fingerprint: PASS
Synthetic Pose-to-features-to-prediction parity: PASS
Contract: PASS
{
  "status": "uncertain",
  "uncertain": true,
  "class_id": null,
  "label": null,
  "label_arabic": null,
  "confidence": 0.257777,
  "top_k": [
    {
      "class_id": 131,
      "label": "itch",
      "label_arabic": "حكة / هرش",
      "score": 0.257777,
      "label_review_status": "unverified",
      "label_conflict": false
    },
    {
      "class_id": 87,
      "label": "nervous system",
      "label_arabic": "جهاز عصبي",
      "score": 0.114222,
      "label_review_status": "unverified",
      "label_conflict": false
    },
    {
      "class_id": 268,
      "label": "treason",
      "label_arabic": "خيانة",
      "score": 0.064608,
      "label_review_status": "unverified",
      "label_conflict": false
    }
  ],
  "model_version": "karsl_cnn_502_pass_v1",
  "calibration_version": "cnn502_validation_temperature_v1"
}
Real Pose decoding and predict